# 04 — MCP Prompts

Prompts are **user-controlled** primitives — triggered by user actions like slash commands.

## Learning Objectives

- Understand why servers define prompts
- Create prompt templates with arguments
- Use prompts from a client
- Build slash command workflows

## Why Server-Defined Prompts?

Instead of users writing their own prompts (which may be suboptimal), MCP servers provide **expert-crafted templates**:

| User writes manually | Server provides |
|---------------------|-----------------|
| "rewrite this in markdown" | `/format doc_id` — optimized prompt with tool instructions |
| "summarize that document" | `/summarize doc_id` — tested prompt with specific output format |
| "review my code" | `/review file_path` — comprehensive review checklist |

In [ ]:
# Simulating prompt definitions
# In a real server, these use @mcp.prompt decorator

docs = {
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "plan.md": "The plan outlines the steps for implementation.",
}

## Defining a Format Prompt

In [ ]:
def format_prompt(doc_id: str) -> list[dict]:
    """Generate a prompt that asks Claude to reformat a document in Markdown."""
    prompt_text = f"""
    Your goal is to reformat a document to be written with markdown syntax.

    The id of the document you need to reformat is:
    <document_id>
    {doc_id}
    </document_id>

    Add in headers, bullet points, tables, etc as necessary.
    Use the 'edit_document' tool to edit the document.
    After the document has been edited, respond with the final version.
    Don't explain your changes.
    """
    return [{"role": "user", "content": prompt_text}]

# Generate the prompt for a specific document
messages = format_prompt("report.pdf")
print("Generated prompt messages:")
for msg in messages:
    print(f"  Role: {msg['role']}")
    print(f"  Content: {msg['content'][:100]}...")

## Defining a Summarize Prompt

In [ ]:
def summarize_prompt(doc_id: str) -> list[dict]:
    """Generate a prompt that asks Claude to summarize a document."""
    prompt_text = f"""
    Read the document with id '{doc_id}' using the read_doc_contents tool.
    Then provide a concise summary in 2-3 sentences.
    Focus on the key points and main takeaways.
    """
    return [{"role": "user", "content": prompt_text}]

messages = summarize_prompt("plan.md")
print("Summarize prompt for plan.md:")
print(messages[0]["content"])

## Real Server Code with Prompts

In [ ]:
PROMPT_SERVER_CODE = '''
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base
from pydantic import Field

mcp = FastMCP("DocumentMCP", log_level="ERROR")

@mcp.prompt(
    name="format",
    description="Rewrites the contents of the document in Markdown format.",
)
def format_document(
    doc_id: str = Field(description="Id of the document to format"),
) -> list[base.Message]:
    prompt = f"""
    Your goal is to reformat a document to be written with markdown syntax.
    The id of the document you need to reformat is:
    <document_id>{doc_id}</document_id>
    Use the 'edit_document' tool to edit the document.
    """
    return [base.UserMessage(prompt)]


@mcp.prompt(
    name="summarize",
    description="Summarize a document concisely.",
)
def summarize_document(
    doc_id: str = Field(description="Id of the document to summarize"),
) -> list[base.Message]:
    return [base.UserMessage(
        f"Read document '{doc_id}' and provide a 2-3 sentence summary."
    )]
'''

print("Key patterns:")
print("  @mcp.prompt(name=..., description=...)")
print("  Function receives arguments (e.g., doc_id)")
print("  Returns list[base.Message] — sent directly to Claude")
print("  base.UserMessage(text) — creates a user message")

## Prompt Workflow

```
User types: /format report.pdf
    │
    ▼
Client parses: command="format", arg="report.pdf"
    │
    ▼
Client calls: get_prompt("format", {"doc_id": "report.pdf"})
    │
    ▼
Server executes: format_document(doc_id="report.pdf")
    │
    ▼
Server returns: [UserMessage("Reformat document 'report.pdf'...")]
    │
    ▼
Client sends messages to Claude
    │
    ▼
Claude uses tools to read, reformat, and save the document
    │
    ▼
User sees the formatted result
```

## Client-Side Prompt Usage

In [ ]:
# Simulating client-side prompt operations

# Available prompts (returned by list_prompts)
available_prompts = [
    {"name": "format", "description": "Rewrites document in Markdown", "arguments": [{"name": "doc_id"}]},
    {"name": "summarize", "description": "Summarize a document", "arguments": [{"name": "doc_id"}]},
]

print("Available prompts (slash commands):")
for prompt in available_prompts:
    args = ", ".join(a["name"] for a in prompt["arguments"])
    print(f"  /{prompt['name']} <{args}> — {prompt['description']}")

print("\nUsage examples:")
print("  /format report.pdf")
print("  /summarize plan.md")

## Exercise: Create a Custom Prompt

Create a `translate` prompt that:
1. Takes a `doc_id` and `language` parameter
2. Asks Claude to translate the document to the specified language
3. Instructs Claude to use the edit_document tool to save the translation

In [ ]:
# TODO: Implement translate_prompt

def translate_prompt(doc_id: str, language: str) -> list[dict]:
    """Generate a prompt to translate a document."""
    prompt_text = f"""
    Read the document '{doc_id}' using the read_doc_contents tool.
    Translate its contents to {language}.
    Use the edit_document tool to replace the original text with the translation.
    Respond with the translated version.
    """
    return [{"role": "user", "content": prompt_text}]

# Test
messages = translate_prompt("report.pdf", "Spanish")
print("Translate prompt:")
print(messages[0]["content"])